# OmniCare Clinical Copilot - Notebook 0: Setup & Models

This notebook sets up the environment, installs dependencies, loads the AI models,
and defines shared utilities used across all other notebooks.

**Models:**
- `google/medgemma-1.5-4b-it` — Medical multimodal LLM (4-bit quantized)
- `openai/whisper-large-v3-turbo` — Speech-to-text ASR

**GPU Budget:** ~5-6 GB on T4 16GB (comfortable headroom)

## 0. Colab Bootstrap (run this first)

Auto-detects environment. In Colab, clones the private repo using your GitHub PAT.

**One-time setup:** In Colab, go to the **Key icon** in the left sidebar > Add a secret named `GITHUB_PAT` with your [GitHub Personal Access Token](https://github.com/settings/tokens).

In [ ]:
# ===========================================================
# Colab Bootstrap - run this cell first (works locally too)
# ===========================================================
import os, sys

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = '/content/omnicare-clinical-copilot'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        token = userdata.get('GITHUB_PAT')
        repo_url = f'https://{token}@github.com/Yashground/omnicare-clinical-copilot.git'
        os.system(f'git clone {repo_url} {REPO_DIR}')
    NOTEBOOKS_DIR = os.path.join(REPO_DIR, 'notebooks')
    os.makedirs('/content/encounters', exist_ok=True)
    os.makedirs('/content/sample_data', exist_ok=True)
else:
    NOTEBOOKS_DIR = os.path.dirname(os.path.abspath('__file__'))

if NOTEBOOKS_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOKS_DIR)

print(f'Environment ready | Colab: {IN_COLAB} | Notebooks dir: {NOTEBOOKS_DIR}')

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch
!pip install -q soundfile librosa
!pip install -q pydicom Pillow
!pip install -q fhir.resources pydantic
!pip install -q huggingface_hub

## 2. Authenticate with HuggingFace

MedGemma is a gated model — you must have accepted the license at
[huggingface.co/google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it).

In [ ]:
from huggingface_hub import login

# This will prompt for your HuggingFace token
# You can also set HF_TOKEN environment variable or use Colab secrets
login()

## 3. Check GPU Availability

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Models will run very slowly on CPU.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU")

## 4. Load MedGemma 1.5-4b-it (4-bit Quantized)

This model handles:
- SOAP note generation from transcripts
- Admission note generation
- Radiology image analysis (multimodal)
- Discharge summary generation

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MEDGEMMA_MODEL_ID = "google/medgemma-1.5-4b-it"

# 4-bit quantization config — fits in T4 16GB GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {MEDGEMMA_MODEL_ID} with 4-bit quantization...")
medgemma_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)

print(f"MedGemma loaded. VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 5. Load Whisper Large v3 Turbo (ASR)

Used for medical audio transcription with vocabulary prompting.

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

WHISPER_MODEL_ID = "openai/whisper-large-v3-turbo"

print(f"Loading {WHISPER_MODEL_ID}...")
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

whisper_processor = AutoProcessor.from_pretrained(WHISPER_MODEL_ID)

asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch.float16,
    device="cuda"
)

print(f"Whisper loaded. Total VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 6. Sanity Check — Test Both Models

In [ ]:
# Test MedGemma with a simple medical question
test_messages = [
    {"role": "user", "content": "What are the common symptoms of pneumonia?"}
]

inputs = medgemma_processor.apply_chat_template(
    test_messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(medgemma_model.device)

with torch.no_grad():
    output_ids = medgemma_model.generate(**inputs, max_new_tokens=200)

new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
response = medgemma_processor.decode(new_tokens, skip_special_tokens=True)
print("MedGemma test response:")
print(response[:500])

In [ ]:
# Test Whisper with a short audio sample (if available)
# For now, just verify the pipeline is functional
print("Whisper ASR pipeline ready.")
print(f"Model: {WHISPER_MODEL_ID}")
print(f"Supports: WAV, MP3, FLAC, OGG audio formats")
print(f"Medical prompting enabled via initial_prompt parameter")

## 7. GPU Memory Summary

In [ ]:
!nvidia-smi

---

**Next:** Run `01_consultation_audio_soap.ipynb` for the audio transcription and SOAP generation pipeline.

**Important:** Keep this notebook's runtime active — the models are loaded in GPU memory
and shared across notebooks via Colab's session state.